In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/konstantineendeladze/basic-clean-done/test_clean.parquet
/kaggle/input/datasets/konstantineendeladze/basic-clean-done/y.csv
/kaggle/input/datasets/konstantineendeladze/basic-clean-done/X_clean.parquet


**Imports**


In [4]:


import numpy as np
import pandas as pd
import os
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

import mlflow
import dagshub

warnings.filterwarnings("ignore")

** Load & merge data**

In [5]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading train transaction...")
train_tr = pd.read_csv(path + "train_transaction.csv")
print(f"  train_transaction: {train_tr.shape}")

print("Loading train identity...")
train_id = pd.read_csv(path + "train_identity.csv")
print(f"  train_identity: {train_id.shape}")

print("Loading test transaction...")
test_tr = pd.read_csv(path + "test_transaction.csv")
print(f"  test_transaction: {test_tr.shape}")

print("Loading test identity...")
test_id = pd.read_csv(path + "test_identity.csv")
print(f"  test_identity: {test_id.shape}")

# ── fix column name mismatch: test uses hyphens, train uses underscores ──
print("\nFixing identity column names (hyphen → underscore)...")
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]
train_id.columns = [c.replace("-", "_") for c in train_id.columns]
print(f"  Sample train_id cols: {list(train_id.columns[:5])}")
print(f"  Sample test_id cols:  {list(test_id.columns[:5])}")

print("\nMerging...")
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"test shape: {test.shape}")

print("\nAligning columns between train and test...")
only_in_train = set(X.columns) - set(test.columns)
only_in_test  = set(test.columns) - set(X.columns)
print(f"  Cols only in train: {len(only_in_train)} → {only_in_train}")
print(f"  Cols only in test:  {len(only_in_test)}  → {only_in_test}")

common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
print(f"\nAfter alignment:")
print(f"  X shape:    {X.shape}")
print(f"  test shape: {test.shape}")
assert list(X.columns) == list(test.columns), "Column mismatch!"
print("Columns aligned ✓")

print(f"\nFraud rate: {y.mean():.4f} ({y.sum()} fraud out of {len(y)} total)")

Loading train transaction...
  train_transaction: (590540, 394)
Loading train identity...
  train_identity: (144233, 41)
Loading test transaction...
  test_transaction: (506691, 393)
Loading test identity...
  test_identity: (141907, 41)

Fixing identity column names (hyphen → underscore)...
  Sample train_id cols: ['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04']
  Sample test_id cols:  ['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04']

Merging...
  train merged: (590540, 434)
  test merged:  (506691, 433)

X shape: (590540, 432)
y shape: (590540,)
test shape: (506691, 432)

Aligning columns between train and test...
  Cols only in train: 0 → set()
  Cols only in test:  0  → set()

After alignment:
  X shape:    (590540, 432)
  test shape: (506691, 432)
Columns aligned ✓

Fraud rate: 0.0350 (20663 fraud out of 590540 total)


In [6]:
print("=== COLUMN REDUCTION ===")
print(f"Starting shape: X={X.shape}, test={test.shape}")

# ── 1. drop high missing value columns (>50% missing) ──
print("\nStep 1: Dropping columns with >50% missing values...")
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.5].index.tolist()
print(f"  Columns to drop: {len(high_missing)}")
print(f"  {high_missing}")

X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  After drop: X={X.shape}, test={test.shape}")

# ── 2. drop near-zero variance columns (numeric only) ──
print("\nStep 2: Dropping near-zero variance numeric columns (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"  Columns to drop: {len(low_var)}")
print(f"  {low_var}")

X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  After drop: X={X.shape}, test={test.shape}")

# ── 3. drop high cardinality categorical columns (>200 unique values) ──
print("\nStep 3: Dropping high cardinality categorical columns (>200 unique values)...")
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"  Columns to drop: {len(high_card)}")
print(f"  {high_card}")

X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)
print(f"  After drop: X={X.shape}, test={test.shape}")

print(f"\n=== FINAL SHAPES ===")
print(f"  X:    {X.shape}")
print(f"  test: {test.shape}")
assert list(X.columns) == list(test.columns), "Column mismatch!"
print("Columns aligned ✓")

# ── memory usage ──
print(f"\nMemory usage:")
print(f"  X:    {X.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  test: {test.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

=== COLUMN REDUCTION ===
Starting shape: X=(590540, 432), test=(506691, 432)

Step 1: Dropping columns with >50% missing values...
  Columns to drop: 214
  ['dist1', 'dist2', 'R_emaildomain', 'D5', 'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14', 'M5', 'M7', 'M8', 'M9', 'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V167', 'V168', 'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176', 'V177', 'V178', 'V179', 'V180', 'V181', 'V182', 'V183', 'V184', 'V185', 'V186', 'V187', 'V188', 'V189', 'V190', 'V191', 'V192', 'V193', 'V194', 'V195', 'V196', 'V197', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V217', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228', 'V229

In [7]:
print("=== MEMORY OPTIMIZATION ===")
print(f"Before: X={X.memory_usage(deep=True).sum()/1024**2:.1f} MB, test={test.memory_usage(deep=True).sum()/1024**2:.1f} MB")

def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32)  # float16 too lossy, use float32
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")

print(f"\nAfter optimization:")
print(f"  X:    {X.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"  test: {test.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nColumn dtypes summary:")
print(X.dtypes.value_counts())

=== MEMORY OPTIMIZATION ===
Before: X=1172.3 MB, test=1008.7 MB
  X: 704.8 MB
  test: 607.7 MB

After optimization:
  X:    704.8 MB
  test: 607.7 MB

Column dtypes summary:
float32    205
object       9
int32        1
int16        1
Name: count, dtype: int64


**MLflow / DagsHub setup**

**Train/val split**

In [8]:
print("=== TRAIN/VAL SPLIT ===")
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"\nTrain fraud rate: {y_train.mean():.4f}")
print(f"Val   fraud rate: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"\nNumeric columns:     {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"Total:               {len(num_cols) + len(cat_cols)}")

print(f"\nMemory after split:")
print(f"  X_train: {X_train.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"  X_val:   {X_val.memory_usage(deep=True).sum()/1024**2:.1f} MB")

# free up X to save RAM since we have X_train and X_val now
del X
import gc
gc.collect()
print("\nFreed X from memory ✓")

=== TRAIN/VAL SPLIT ===
  X_train: (472432, 216), y_train: (472432,)
  X_val:   (118108, 216),   y_val:   (118108,)

Train fraud rate: 0.0350
Val   fraud rate: 0.0350

Numeric columns:     207
Categorical columns: 9
Total:               216

Memory after split:
  X_train: 567.5 MB
  X_val:   141.9 MB

Freed X from memory ✓


Splitting data...
  X_train: (472432, 432), y_train: (472432,)
  X_val:   (118108, 432),   y_val:   (118108,)

Train fraud rate: 0.0350
Val   fraud rate: 0.0350

Numeric columns:     401
Categorical columns: 31
Total:               432


**Build pipeline & train**

In [9]:
print("=== BUILDING PIPELINE ===")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=100,
        class_weight="balanced",
        random_state=42
    ))
])

print("Pipeline built ✓")
print(f"  Numeric features:     {len(num_cols)}")
print(f"  Categorical features: {len(cat_cols)}")
print(f"\nTraining model... (this may take a few minutes)")

model.fit(X_train, y_train)

print("Training done ✓")

# free X_train from memory now that model is trained
del X_train
gc.collect()
print("Freed X_train from memory ✓")

=== BUILDING PIPELINE ===
Pipeline built ✓
  Numeric features:     207
  Categorical features: 9

Training model... (this may take a few minutes)
Training done ✓
Freed X_train from memory ✓


** Evaluate & log to MLflow**

In [10]:
print("=== EVALUATING MODEL ===")

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"\n=== VALIDATION METRICS ===")
print(f"  AUC:       {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

metrics = {
    "val_auc":       val_auc,
    "val_precision": val_precision,
    "val_recall":    val_recall,
    "val_f1":        val_f1,
}

print("\nLogging to MLflow / DagsHub...")
with mlflow.start_run(run_name="decision_tree_baseline"):
    mlflow.log_params({
        "max_depth": 6,
        "min_samples_leaf": 100,
        "class_weight": "balanced",
        "test_size": 0.2,
        "random_state": 42,
        "missing_threshold": 0.5,
        "low_variance_threshold": 0.01
    })
    mlflow.log_metrics(metrics)
    joblib.dump(model, "tree_model.pkl")
    mlflow.log_artifact("tree_model.pkl")
    print(f"  Run ID: {mlflow.active_run().info.run_id}")

print("Logged to DagsHub ✓")
print("Check: https://dagshub.com/kende23/ieee-cis-fraud-detection")

# free val data
del X_val
gc.collect()
print("Freed X_val from memory ✓")

=== EVALUATING MODEL ===

=== VALIDATION METRICS ===
  AUC:       0.8458
  Precision: 0.1355
  Recall:    0.7237
  F1:        0.2282

Logging to MLflow / DagsHub...


2026/05/02 13:21:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/02 13:21:45 INFO mlflow.store.db.utils: Updating database tables


  Run ID: 156fe3d7b97e48308965cd0d39330b16
Logged to DagsHub ✓
Check: https://dagshub.com/kende23/ieee-cis-fraud-detection
Freed X_val from memory ✓


**Make submission**

In [11]:
print("=== GENERATING SUBMISSION ===")
print(f"  test shape: {test.shape}")
print(f"  Columns match: {list(test.columns) == list(model.named_steps['preprocessor'].feature_names_in_.tolist()) if hasattr(model.named_steps['preprocessor'], 'feature_names_in_') else 'check skipped'}")

print("\nPredicting...")
test_probs = model.predict_proba(test)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Predicted fraud rate: {test_probs.mean():.4f}")
print(f"  Min prob: {test_probs.min():.4f}")
print(f"  Max prob: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Sample:\n{submission.head(10)}")

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION ===
  test shape: (506691, 216)
  Columns match: True

Predicting...
  Predictions shape: (506691,)
  Predicted fraud rate: 0.3276
  Min prob: 0.0000
  Max prob: 0.9611

Submission shape: (506691, 2)
Sample:
   TransactionID   isFraud
0        3663549  0.277075
1        3663550  0.277075
2        3663551  0.277075
3        3663552  0.097099
4        3663553  0.090969
5        3663554  0.090969
6        3663555  0.568091
7        3663556  0.470201
8        3663557  0.090969
9        3663558  0.277075

submission.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
